# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset loaded:\n------------------")
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

* **Record sets and their `@id`:**
* **Fields and columns** can be referenced by their `@id` values.


In [ ]:
# List all record sets and their meta-information
pp = pprint.PrettyPrinter(indent=2)
record_sets = list(dataset.record_sets())
print("Record sets in the dataset:\n---------------------------")
for rs in record_sets:
    print(f"@id: {rs['@id']}")
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    # List fields (columns) for each record set
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields/columns (@id, name):")
        for f in fields:
            print(f"    - {f['@id']}: {f.get('name', '')}")
    print("----------------------------")

# Optionally, display the first few records for the main table (using the main record set @id)
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nSample records for record set: {main_record_set_id}")
    sample_records = list(dataset.records(record_set=main_record_set_id))[:3]
    pp.pprint(sample_records)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Collect all record set ids from the dataset
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        # Avoid empty dataframes with only NaN columns
        if not df.empty:
            dataframes[record_set_id] = df

# Show columns from the first record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns for main record set (@id): {main_record_set_id}\n------------------------------------------")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nTop 5 rows:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Use the column names obtained previously. If unsure, print the columns again.
df = dataframes[main_record_set_id]
print(f"Available columns in '{main_record_set_id}':\n", df.columns.tolist())

# Choose a numeric field (common for protein data: 'MW' for molecular weight, or a peptide count field)
# Adjust field names as appropriate for the actual dataset (see printout above for actual names)

# Try to select a field for 'MW' (molecular weight) or fallback to the first numeric column
import numpy as np
potential_numeric_fields = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (int, float, np.number))).any()]
if 'MW' in df.columns:
    numeric_field_id = 'MW'
else:
    numeric_field_id = potential_numeric_fields[0] if potential_numeric_fields else df.columns[0]

print(f"Using numeric field for analysis: {numeric_field_id}")

# Filter: Remove non-numeric or missing values
df_num = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id])

threshold = df_num[numeric_field_id].quantile(0.75)  # Top quartile
filtered_df = df_num[df_num[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} records\n")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Suggest a group field: e.g. 'Sample', 'Protein Family', etc. Fallback to non-numeric field
candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field_id]
group_field = candidate_group_fields[0] if candidate_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped {numeric_field_id} mean by '{group_field}':")
    display(grouped_df.to_frame().head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df_num[numeric_field_id], bins=30, kde=True, color='skyblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

if group_field:
    # Boxplot by group (if grouping field exists)
    plt.figure(figsize=(12,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Through this notebook, we've loaded the FAIR^2 mass spectrometry dataset using the Croissant schema and `mlcroissant`. We reviewed schema record sets, explored main numeric fields such as molecular weight (MW), performed basic filtering and normalization, grouped by categorical attributes, and visualized the distributions. This approach facilitates transparent, reproducible biomedical data science in accordance with FAIR principles.

*You are encouraged to extend this analysis according to your research goals—such as by exploring additional record sets, integrating computational tools for modification or peptide sequence analysis, or leveraging the rich metadata documented in the Croissant schema.*